In [ ]:
import os
DATA_FOLDER =r"E:\Github\uit_chatbot\graph\data"
file_names = []

for root, dirs, files in os.walk(DATA_FOLDER):
    for file in files:
        if file.lower().endswith((".pdf", ".doc", ".docx")):
            file_names.append(os.path.join(root, file))

print(len(file_names))
for file_name in file_names:
    print(file_name)

In [ ]:
import fitz

pdf_path = file_names[1]
print(pdf_path)
doc = fitz.open(pdf_path)
print("Number of pages:", doc.page_count)

text = ""
for page in doc:
    text += page.get_text("text") + "\n"

print(text)

In [ ]:
page = doc.load_page(0)
text = page.get_text("text")

if text.strip():
    print("✅ Text detected:")
    print(text[:500])  # preview first 500 chars
else:
    print("⚠️ No text found — this PDF is likely a scanned image.")


In [ ]:
import fitz
from PIL import Image
import pytesseract

pdf_path = file_names[2]
print(pdf_path)

doc = fitz.open(pdf_path)
text = ""
for page in doc:
    pix = page.get_pixmap()
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    text += pytesseract.image_to_string(img, lang="vie") + "\n"

print(text)

In [ ]:
from sentence_transformers import SentenceTransformer

# INPUT TEXT MUST BE ALREADY WORD-SEGMENTED!
sentences = ["Cô ấy là một người vui_tính .", "Cô ấy cười nói suốt cả ngày ."]

model = SentenceTransformer('bkai-foundation-models/vietnamese-bi-encoder')
embeddings = model.encode(sentences)
print(embeddings)

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

In [4]:
from graph.src.db import *
from bson import ObjectId

# Assuming your classes are already defined above

def test_insert():
    client = init_mongo()
    if not client:
        return

    db = client["kg_test_db"]  # use a test database
    concepts_collection = db["concepts"]
    relations_collection = db["relations"]
    triplets_collection = db["triplets"]

    # Create a Document
    doc1 = Document(_id=str(ObjectId()), document_number="DOC-001")

    # Create a Concept
    concept1 = Concept(name="Physics", documents=[doc1], synonym=["Natural Science"])
    concept_dict = concept1.to_dict()

    # Insert Concept
    concept_result = concepts_collection.insert_one(concept_dict)
    print(f"Inserted Concept with ID: {concept_result.inserted_id}")

    # Create a Relation
    relation1 = Relation(name="studies", documents=[doc1])
    relation_dict = relation1.to_dict()

    # Insert Relation
    relation_result = relations_collection.insert_one(relation_dict)
    print(f"Inserted Relation with ID: {relation_result.inserted_id}")

    # Create a Triplet
    triplet1 = Triplet(subject_id=concept_result.inserted_id, relation_id=relation_result.inserted_id, object_id=concept_result.inserted_id)
    triplet_result = triplets_collection.insert_one(triplet1.to_dict())
    print(f"Inserted Triplet with ID: {triplet_result.inserted_id}")

In [3]:
test_insert()

You successfully connected to MongoDB!
Inserted Concept with ID: 690ef9865fcecdba99c7ff75
Inserted Relation with ID: 690ef9865fcecdba99c7ff76
Inserted Triplet with ID: 690ef9865fcecdba99c7ff77


In [6]:
from graph.src.triplet_extraction import clean_text

sentence = '"kiểm tra công trình đường bộ bao gồm kiểm tra theo quy chuẩn, tiêu chuẩn kỹ thuật, quy trình bảo trì được duyệt"'
sentence = clean_text(sentence)
print(sentence)

kiểm tra công trình đường bộ bao gồm kiểm tra theo quy chuẩn, tiêu chuẩn kỹ thuật, quy trình bảo trì được duyệt
